In [2]:
import numpy as np
import numpy.linalg as lin

np.set_printoptions(precision=3, linewidth=150, suppress=True)

## Systèmes matriciels

Un système de plusieurs équations à autant d'inconnues peut s'écrire sous forme d'un système matriciel de la forme $A {\bf x} = {\bf b}$ :

$$
\begin{bmatrix}
a_{11} a_{12} \ldots a_{1n} \\
a_{21} a_{22} \ldots a_{2n} \\
 \vdots \\
a_{n1} a_{n2} \ldots a_{nn} \\
\end{bmatrix}
\;
\begin{bmatrix}
x_{1} \\
x_{2} \\
\vdots \\
x_{n} \\
\end{bmatrix}
=
\begin{bmatrix}
b_{1} \\
b_{2} \\
\vdots \\
b_{n} \\
\end{bmatrix}
$$

Pour résourdre ce dernier la facon intuitive est d'inverser $A$ et de calculer $A^{-1}\, {\bf b}$ :

In [3]:
A = np.array([[6,5,4], [5,3,2], [7,3,2]])
b = np.array([11.7, 7.9, 9.5])
print(A, b)

x = lin.inv(A) @ b
x

[[6 5 4]
 [5 3 2]
 [7 3 2]] [11.7  7.9  9.5]


array([0.8, 0.9, 0.6])

## 1. Méthode du pivot de Gauss

On transforme A en une matrice triangulaire qui permet ensuite de résoudre le système en O(n²) opérations.

Pour cela on commence à mettre des 0 sur la première colonne en dessous de la diagonale. Pour cela il suffit de multiplier $A$ par la matrice $E_1$ suivante, *à gauche* :

$$
E_1 = 
\begin{bmatrix}
\;1 \quad 0\; 0 \ldots 0 \\
\frac{-a_{21}}{a_{11}} \, 1\; 0  \ldots 0 \\
\frac{-a_{31}}{a_{11}} \, 0\; 1  \ldots 0 \\
\vdots \\
\frac{-a_{n1}}{a_{11}}\; 0\; 0  \ldots 1 \\
\end{bmatrix}
$$

$E_2$ sera la matrice équivalente avec des termes $\frac{-a_{k2}}{a_{22}}$ sous la diagonale afin des 0 dans la 2e colonne de $A$ sous la diagonale, etc.

Bien sûr si on multiplie $A$ par $E_1$, il faut aussi multiplier ${\bf b}$ par $E_1$ pour que le système matriciel soit équivalent. Cette multiplication $E_1 \, {\bf b}$ peut se calculer nettement plus rapidement qu'avec un produit matrice vecteur.

#### Système matriciel avec matrice triangulaire

Pour finir il faut résoudre $U {\bf x} = {\bf c}$ avec $U$ une matrice triangulaire supérieur. Cela se fait en partant de la dernière ligne qui donne directement `x[-1] = c[-1] / U[-1,-1]`. Une fois `x[-1]` connu, on en déduit la valeur de `x[-2]` avec l'avant dernière ligne, etc.

En faisant les calculs on constate qu'il faut environ n²/2 additions et multiplications.

In [4]:
# Cf analyse du code ci-dessous

def solve_gauss(A, b):
    A = A.astype(np.float64)   # si A a des entiers, on va avoir des erreurs de calculs
    for i in range(len(A)-1):
        coefs = - A[i+1:,i] / A[i,i]  # Normalement il faut tester que A[i,i] != 0
        A[i+1:, i:] += np.outer(coefs, A[i, i:])
        b[i+1:] += coefs * b[i]
    # A est maintenant triangulaire supérieur
    res = np.zeros(len(b), dtype=b.dtype)
    res[-1] = b[-1] / A[-1,-1]
    for i in range(len(A)-1)[::-1]:
        res[i] = (b[i] - A[i,i+1:] @ res[i+1:]) / A[i,i]
    return res


#### Exercice

1. S'assurer de bien comprendre le code ci-dessus. (Insérer des `print` pour voir ce qui se passe.)

1. C'est quoi `coefs` (ligne 4) ?
_Ça correspond à la colonne $i$ de la matrice $E_i-I$_

1. Que fait la ligne 6 (`b[i+1:] += coefs * b[i]`) ?
_Elle calcule $E_i {\bf b} = (I+\textbf{Coefs})\, {\bf b} = {\bf b} + \textbf{Coefs}\, {\bf b}$_

1. Que fait la ligne 5 (`A[i+1:, i:] += np.outer(coefs, A[i, i:])`) ?
_Elle calcule $E_i A = A + \textbf{Coefs} A$, voir ci-dessous_

#### Réponse à la question 4

La ligne 5 calcule $E_i A = A + \textbf{Coefs} A$.

Le problème est que `Coefs @ A` fait n³ opérations ce qui est beaucoup trop et surtout inutile sachant que **Coefs** est un vecteur dans une matrice de zéros. Si ce vecteur est dans la colonne $i$ alors seuls les termes de la ligne $i$ de A serviront pour générer ce produit matriciel. De plus on a `res[j,k] = Coef[j,i] * A[i,k]` ce qui s'écrit en Numpy `res = np.outer(coefs, A[i,:])` si `coefs` est la colonne $i$ de `Coefs`.

Comme la matrice **Coefs** n'a des valeurs non nulles qu'en dessous de la diagonale, cela veut dire que le résultat `res` de la multiplication n'a que des zéros au dessus de la ligne $i$. Donc autant ne faire que le calcul pour ce qui est en
dessus de la ligne $i$ : `A[i+1:, i:] += np.outer(coefs, A[i, i:])` avec `coefs` un vecteur de taille $n-i$. 

In [5]:
A = 10 * np.random.random(size=(5,5))
b = A.sum(axis=1)
print(f"A\n {A} \nb\n {b}")
solve_gauss(A, b)

A
 [[5.07  8.896 4.161 0.555 0.62 ]
 [9.969 9.33  4.277 6.322 4.845]
 [5.762 7.417 2.704 9.331 5.299]
 [9.277 4.558 1.184 8.309 6.86 ]
 [1.567 7.158 0.192 9.173 3.913]] 
b
 [19.301 34.744 30.512 30.188 22.003]


array([1., 1., 1., 1., 1.])

## 2. Méthode du pivot de Gauss partiel

On a vu dans le cours que la méthode de Gauss peut induire des erreurs d'arrondi :

In [6]:
np.set_printoptions(precision=20, linewidth=150, suppress=True)
np.finfo(np.float32)

finfo(resolution=1e-06, min=-3.4028235e+38, max=3.4028235e+38, dtype=float32)

In [7]:
e = 1E-6
A = np.array([[e, 1], [1, 2]], dtype='float32')
b = np.array([1., 3.], dtype='float32')
print(f"A\n {A} \nb\n {b}")

A
 [[0.000001 1.      ]
 [1.       2.      ]] 
b
 [1. 3.]


In [8]:
x = solve_gauss(A.copy(),b)
print("x =", x)
A @ x                # on ne retrouve pas exactement b

x = [1.013279 0.999999]


array([1.      , 3.013277], dtype=float32)

Il y a un problème d'arrondi évident sachant que la solution est [1 + 2*e, 1 - e] :

In [9]:
x = np.array([1 + 2*e, 1 - e], dtype='float32')
print("La solution est :", x)
A @ x             # on retrouve b

La solution est : [1.000002 0.999999]


array([1., 3.], dtype=float32)

Le problème est qu'on divise les valeurs de la première colonne de A par le premier pivot : 1E-6, ce qui nous fait perdre de la précision.
La **méthode du pivot partiel** essaye de résoudre ce problème en échangeant des lignes à chaque itération, pour trouver le pivot de plus grande valeur absolue, avant de faire le calcul $E_i A$.
(La méthode du pivot _total_ échange aussi des colonnes, mais c'est plus compliqué.)

La structure de la matrice A après 3 itérations ressemble à cela :

```
 ----------------
  \              |
    \            |
      \          |
       |         |
   0   |         |
       |         |
       -----------
```
Aussi il est important que ne pas détruire le travail déjà fait à savoir de garder les 0 sous la diagonale.
Cela implique qu'on ne doit pas changer la 4e ligne avec une ligne au dessus. On ne regarde donc que les valeurs de la 4e colonne qui sont _en dessous de la diagonale_ pour choisir le nouveau pivot.

#### Exercice

Améliorer la fonction `solve_gauss` en implémentant la méthode de pivot partiel.
(La fonction `np.argmax` peut vous être utile.)

In [10]:
def solve_gauss_partial(A, b):
    A = A.astype(np.float64)   # si A a des entiers, on va avoir des erreurs de calculs
    for i in range(len(A)-1):
        # begin new code
        pivot = i + np.argmax(np.abs(A[i:, i]))
        A[[i, pivot]] = A[[pivot, i]]
        b[[i, pivot]] = b[[pivot, i]]
        # end new code
        coefs = - A[i+1:,i] / A[i,i]  # Normalement il faut tester que A[i,i] != 0
        A[i+1:, i:] += np.outer(coefs, A[i, i:])
        b[i+1:] += coefs * b[i]
    # A est maintenant triangulaire supérieur
    res = np.zeros(len(b), dtype=b.dtype)
    res[-1] = b[-1] / A[-1,-1]
    for i in range(len(A)-1)[::-1]:
        res[i] = (b[i] - A[i,i+1:] @ res[i+1:]) / A[i,i]
    return res


In [11]:
e = 1E-6
A = np.array([[e, 1], [1, 2]], dtype='float32')
b = np.array([1., 3.], dtype='float32')
print(f"A\n {A} \nb\n {b}\n")
x = solve_gauss_partial(A, b)
print('solution : ',x)  # c'est bon, maintenant on a trouvé la bonne solution
print('vérification\n', A@x)

A
 [[0.000001 1.      ]
 [1.       2.      ]] 
b
 [1. 3.]

solution :  [1.0000019  0.99999905]
vérification
 [1. 3.]


## 3. Méthode de Jacobi

Si on travaille avec des matrices de grande taille (n égal à 1 milliard, par exemple),
la méthode de Gauss devient trop lente.
(C'est quoi n²/2 dans ce cas ?)

À la place on utilise des méthodes itératives pour _s'approcher_ pas à pas de la solution.
On arrête le calcul lorsqu'on estime qu'on est à une distance choisie de la solution (qu'on appelle l'_erreur_) plutôt que d'attendre d'être à la solution exacte.
(De toute facon la notion de solution exacte sur un ordinateur qui est limité en précision est doutable. _On cherchera donc jamais à avoir une erreur plus petite que notre précision maximale_.)

L'idée fondamentale de l'algorithme itératif est d'avoir une formule du type
$\; {\bf x}^{t+1} = N \, {\bf x}^t + {\bf b}\;$ ou en Python:

```
x = np.random(size = b.size)
while np.square(x - old_x) > seuil:
    old_x = x
    x = N @ x + b
```

La magie c'est si **x** converge. Dans ce cas on a atteint un point fixe,
c. à d. que ${\bf x}^{t+1} = {\bf x}^t$ et donc 

$${\bf x}^t = N\, {\bf x}^t + {\bf b} \quad\text{c. à d.}\quad (I-N)\, {\bf x}^t = {\bf b}$$

On retrouve notre $A \; {\bf x} = {\bf b}$ en posant $N=I-A$.

### Méthode de Jacobi

La méthode de Jacobi n'utilise pas $N=I-A$ car en général cela ne converge pas.
À la place on découpe la matrice $A$ en $M$ et $N$ avec

* $M$ la matrice diagonale qui comprend les éléments de la diagonale de $A$
* $N = M - A$  (donc 0 sur la diagonale et l'opposé des éléments de $A$ ailleurs)

Ainsi le système à résoudre est $\; (M - N) \, {\bf x} = {\bf b}$.

La formule itérative est donc pour l'itération $k+1$ exprimée en fonction de l'itération $k$ :

$$
M \; {\bf x}^{k+1} =  N\; {\bf x}^k + {\bf b}
$$

et comme $M$ est une matrice diagonale :

$$
\begin{array}{l}
a_{11} x_1^{k+1} = \qquad\quad -a_{12} \, x_2^k - a_{13} \, x_3^k  \ldots - a_{1n} \, x_n^k + b_1 \\
a_{22} x_2^{k+1} = -a_{21} \, x_1^k \qquad\quad - a_{23} \, x_3^k  \ldots - a_{2n} \, x_n^k + b_2 \\
a_{33} x_3^{k+1} = -a_{31} \, x_1^k - a_{32} \, x_2^k  \qquad\quad \ldots - a_{3n} \, x_n^k + b_3 \\
 \vdots \\
a_{nn} x_n^{k+1} = -a_{n1} \, x_1^k - a_{n2} \, x_2^k  \quad \ldots - a_{n-1,n-1} \, x_{n-1}^k + b_n \\
\end{array}
$$

On voit en filigramme $A \; {\bf x} = {\bf b}$.

Pour calculer $x_1^{k+1}$ il faut diviser le terme de droite de la première ligne par $a_{11}$ il faut donc que  $a_{11} \ne 0$.
Pour calculer les termes suivants $x_i^{k+1}$ il faut aussi que $a_{ii}$ soient non nul donc
_il faut que $A$ n'ait pas pas de zéro sur sa diagonale_.

Cela se retrouve dans l'écriture suivante qui reprend la formule initiale :
${\bf x}^{k+1} =  M^{-1} \;(N\; {\bf x}^k + {\bf b})$.
On voit que pour être efficace, il faut que $M$ soit facilement inversible, sinon autant inverser $A$ directement.
Ici c'est bien le cas puisque $M$ est diagonale.

#### Exercice

Programmer Jacobi. (La fonction `np.diag` peut vous être utile ; mais attention `diag` retourne un vecteur quand on l'applique à une matrice, et une matrice quand on l'applique à un vecteur.)

In [3]:
def solve_jacobi(A, b, num_iter=15, verbose=False):
    A = A.astype(np.float64)   # si A a des entiers, on va avoir des erreurs de calculs
    M = np.diag(A)  # attention, c'est un vecteur ; mais ça nous arrangera plus tard
    N = np.diag(M) - A
    print(f"M:\n {np.diag(M)}\nN:\n {N}\n")
    x = np.random.random(4)
    for i in range(num_iter):
        if verbose: print(f"x_{i} = {x}")
        x = (N @ x + b) / M  # on peut faire /M ici au lieu de @inv(M) parce que M est un vecteur
    return x
        
A = np.array([[24,2,4,8], [0,24,9,3], [4,6,16,5], [6,2,1,32]])
b = A.sum(axis=1)  # ainsi la solution sera [1,1,1,1]
print('A:\n', A, "\nb:\n", b, "\n")
solve_jacobi(A, b, verbose=True)

A:
 [[24  2  4  8]
 [ 0 24  9  3]
 [ 4  6 16  5]
 [ 6  2  1 32]] 
b:
 [38 36 31 41] 

M:
 [[24.  0.  0.  0.]
 [ 0. 24.  0.  0.]
 [ 0.  0. 16.  0.]
 [ 0.  0.  0. 32.]]
N:
 [[ 0. -2. -4. -8.]
 [ 0.  0. -9. -3.]
 [-4. -6.  0. -5.]
 [-6. -2. -1.  0.]]

x_0 = [0.841 0.885 0.975 0.45 ]
x_1 = [1.197 1.078 1.255 1.038]
x_2 = [0.938 0.9   0.91  0.95 ]
x_3 = [1.04  1.04  1.069 1.021]
x_4 = [0.978 0.972 0.969 0.988]
x_5 = [1.012 1.013 1.02  1.007]
x_6 = [0.993 0.992 0.99  0.996]
x_7 = [1.004 1.004 1.006 1.002]
x_8 = [0.998 0.998 0.997 0.999]
x_9 = [1.001 1.001 1.002 1.001]
x_10 = [0.999 0.999 0.999 1.   ]
x_11 = [1.    1.    1.001 1.   ]
x_12 = [1. 1. 1. 1.]
x_13 = [1. 1. 1. 1.]
x_14 = [1. 1. 1. 1.]


array([1., 1., 1., 1.])

En essayant avec d'autres matrices, on se rend rapidemment compte que la convergence n'est pas toujours assurée.
Ici ça a marché parce que $A$ est _à diagonale dominante_, on en reparlera en cours.